In [4]:
import torch
import soundfile as sf
import numpy as np
from cli.SparkTTS import SparkTTS
import re

In [5]:
def initialize_model(model_dir="pretrained_models/Spark-TTS-0.5B", device=0):
    """Load the model once at the beginning."""
    device = torch.device(f"cuda:{device}")
    model = SparkTTS(model_dir, device)
    return model


In [6]:
model = initialize_model('/models/Spark-TTS-0.5B/')

/opt/conda/envs/sparktts/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


In [7]:
print(model.model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(166000, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [12]:
def make_prompt(model, text, prompt_audio_path, prompt_text=None):

    # Perform inference and save the output audio
    with torch.no_grad():
        model_prompt, global_token_ids = model.process_prompt(
                text, prompt_audio_path, prompt_text
            )
    

    start_idx = model_prompt.find("<|start_global_token|>")
    voice_params = model_prompt[start_idx:]
    model_prompt = model_prompt[:start_idx]

    pattern = r"<\|start_global_token\|>(.*?)<\|end_global_token\|>"
    match = re.search(pattern, voice_params, re.DOTALL)
    content = match.group(1)
    ids = re.findall(r"<\|bicodec_global_(\d+)\|>", content)
    int_ids = [int(i) for i in ids]
    print(int_ids)

    return model_prompt, int_ids

In [13]:
text = "Hi~, I am Kai from FlashIntel. I am reaching out because we found that you have visited our website recently. Are you available for a quick chat?"
prompt_audio_path = "assets/Ben_promptvn.wav"
model_prompt, int_ids = make_prompt(model, text, prompt_audio_path)

tensor([[[1239, 3377, 1464, 3601, 1315, 1435, 2486,  371, 3149, 2070, 1269,
          1326,  531, 3624, 3196, 1303, 1137, 1320, 2948,  315,  194,  890,
           252,  201, 1461,  794, 3633, 2196,   99, 2624, 2091, 3163]]],
       device='cuda:0', dtype=torch.int32)
[1239, 3377, 1464, 3601, 1315, 1435, 2486, 371, 3149, 2070, 1269, 1326, 531, 3624, 3196, 1303, 1137, 1320, 2948, 315, 194, 890, 252, 201, 1461, 794, 3633, 2196, 99, 2624, 2091, 3163]


In [18]:
full_audio = model.inference_prompt(text, int_ids)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [20]:
full_audio.shape

(130880,)

In [21]:
# 计算每一层的参数数量
def count_parameters(model):
    total_params = 0
    layer_params = {}

    for name, param in model.named_parameters():
        num_params = param.numel()  # 获取当前参数的数量
        total_params += num_params

        # 按层保存每一层的参数量
        layer_params[name] = num_params

    return total_params, layer_params

In [22]:
total_params, layer_params = count_parameters(model.model)

In [23]:
total_params

506634112

In [24]:
for layer, num_params in layer_params.items():
    print(f"Layer: {layer}, Parameters: {num_params}")

Layer: model.embed_tokens.weight, Parameters: 148736000
Layer: model.layers.0.self_attn.q_proj.weight, Parameters: 802816
Layer: model.layers.0.self_attn.q_proj.bias, Parameters: 896
Layer: model.layers.0.self_attn.k_proj.weight, Parameters: 114688
Layer: model.layers.0.self_attn.k_proj.bias, Parameters: 128
Layer: model.layers.0.self_attn.v_proj.weight, Parameters: 114688
Layer: model.layers.0.self_attn.v_proj.bias, Parameters: 128
Layer: model.layers.0.self_attn.o_proj.weight, Parameters: 802816
Layer: model.layers.0.mlp.gate_proj.weight, Parameters: 4358144
Layer: model.layers.0.mlp.up_proj.weight, Parameters: 4358144
Layer: model.layers.0.mlp.down_proj.weight, Parameters: 4358144
Layer: model.layers.0.input_layernorm.weight, Parameters: 896
Layer: model.layers.0.post_attention_layernorm.weight, Parameters: 896
Layer: model.layers.1.self_attn.q_proj.weight, Parameters: 802816
Layer: model.layers.1.self_attn.q_proj.bias, Parameters: 896
Layer: model.layers.1.self_attn.k_proj.weight, 

In [25]:
! pip install tensorboard

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 204.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 225.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 217.9 MB/s eta 0:00:00


In [27]:
from torch.utils.tensorboard import SummaryWriter